# Day 7 & 8: Building the Transformer from Scratch
## The Architecture that Changed AI
---
**Goal:** Build a full Transformer Encoder (similar to the logic in BERT) and train it on the 20 Newsgroups dataset for text classification.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cuda


### 1. The Real-World Dataset
We use `fetch_20newsgroups`. We will focus on 4 distinct categories to keep training fast but realistic.

In [2]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))

X_raw = newsgroups.data
y_raw = newsgroups.target

print(f"Total Samples: {len(X_raw)}")
print(f"Sample text:\n {X_raw[0][:200]}...")

Total Samples: 2257
Sample text:
 Does anyone know of a good way (standard PC application/PD utility) to
convert tif/img/tga files into LaserJet III format.  We would also like to
do the same, converting to HPGL (HP plotter) files.

P...


### 2. Manual Tokenizer & Vocabulary
A Transformer doesn't see words; it sees indices. We build a vocabulary that only keeps the top 5000 most frequent words.

In [3]:
from collections import Counter
import re

def clean_text(text):
    return re.sub(r'[^a-zA-Z\s]', '', text.lower())

all_words = " ".join([clean_text(t) for t in X_raw]).split()
vocab_counts = Counter(all_words)
most_common = vocab_counts.most_common(5000)
vocab = {word: i+2 for i, (word, _) in enumerate(most_common)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def tokenize(text, max_len=128):
    tokens = [vocab.get(w, 1) for w in clean_text(text).split()]
    if len(tokens) < max_len: tokens += [0] * (max_len - len(tokens))
    return tokens[:max_len]

X_tokenized = torch.tensor([tokenize(t) for t in X_raw])
y_labels = torch.tensor(y_raw)

X_train, X_test, y_train, y_test = train_test_split(X_tokenized, y_labels, test_size=0.2)
print(f"X_train shape: {X_train.shape}")

X_train shape: torch.Size([1805, 128])


### 3. Positional Encoding
Since Transformers process all words at once (unlike RNNs), they don't know the order. We inject a 'Sine-Cosine' signal into the embeddings.

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: [Batch, Seq_Len, d_model]
        return x + self.pe[:, :x.size(1)]

#

### 4. Multi-Head Attention (MHA)
This is the core. We split the embedding into multiple 'heads' so the model can attend to different words simultaneously.

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        
    def forward(self, q, k, v):
        bs = q.size(0)
        
        # 1. Linear projection & Split into heads
        Q = self.q_linear(q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(k).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(v).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # 2. Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        
        # 3. Concatenate heads
        concat = torch.matmul(attn, V).transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_k)
        return self.out_linear(concat)

#

### 5. The Transformer Encoder Block
A block contains: Attention -> Add & Norm -> FeedForward -> Add & Norm.

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Sublayer 1: Attention + Residual
        attn_out = self.attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Sublayer 2: FeedForward + Residual
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

### 6. The Final Model: Transformer Classifier
We stack 2 Transformer blocks and use the 'Mean' of the outputs for classification.

In [7]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_classes, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe = PositionalEncoding(d_model)
        
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        
        self.classifier = nn.Linear(d_model, n_classes)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pe(x)
        
        for layer in self.layers:
            x = layer(x)
            
        # Global Average Pooling over the sequence length
        x = x.mean(dim=1)
        return self.classifier(x)

model = TransformerClassifier(vocab_size=5002, d_model=128, n_heads=8, n_classes=4).to(device)
print(model)

TransformerClassifier(
  (embedding): Embedding(5002, 128)
  (pe): PositionalEncoding()
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attention): MultiHeadAttention(
        (q_linear): Linear(in_features=128, out_features=128, bias=True)
        (k_linear): Linear(in_features=128, out_features=128, bias=True)
        (v_linear): Linear(in_features=128, out_features=128, bias=True)
        (out_linear): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=128, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (classifier): Linear(in_features=128, out_features=4, bias=True)
)


### 7. Training on Real Data
This loop will handle the noisy text from the newsgroups.

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()

train_data = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)

model.train()
for epoch in range(50):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

Epoch 1 | Loss: 0.0538
Epoch 2 | Loss: 0.0487
Epoch 3 | Loss: 0.0386
Epoch 4 | Loss: 0.0386
Epoch 5 | Loss: 0.0420
Epoch 6 | Loss: 0.0357
Epoch 7 | Loss: 0.0344
Epoch 8 | Loss: 0.0355
Epoch 9 | Loss: 0.0335
Epoch 10 | Loss: 0.0342
Epoch 11 | Loss: 0.0361
Epoch 12 | Loss: 0.0340
Epoch 13 | Loss: 0.0341
Epoch 14 | Loss: 0.0328
Epoch 15 | Loss: 0.0324
Epoch 16 | Loss: 0.0328
Epoch 17 | Loss: 0.0327
Epoch 18 | Loss: 0.0323
Epoch 19 | Loss: 0.0326
Epoch 20 | Loss: 0.0648
Epoch 21 | Loss: 0.3226
Epoch 22 | Loss: 0.1294
Epoch 23 | Loss: 0.0565
Epoch 24 | Loss: 0.0335
Epoch 25 | Loss: 0.0334
Epoch 26 | Loss: 0.0339
Epoch 27 | Loss: 0.0328
Epoch 28 | Loss: 0.0322
Epoch 29 | Loss: 0.0324
Epoch 30 | Loss: 0.0324
Epoch 31 | Loss: 0.0320
Epoch 32 | Loss: 0.0359
Epoch 33 | Loss: 0.0328
Epoch 34 | Loss: 0.0336
Epoch 35 | Loss: 0.0339
Epoch 36 | Loss: 0.0334
Epoch 37 | Loss: 0.0329
Epoch 38 | Loss: 0.0326
Epoch 39 | Loss: 0.0318
Epoch 40 | Loss: 0.0341
Epoch 41 | Loss: 0.0325
Epoch 42 | Loss: 0.0320
E

### 8. Evaluating Real-World Performance
How well does our 'Hardcoded' Transformer handle unseen news posts?

In [13]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test.to(device))
    predictions = test_outputs.argmax(dim=1)
    accuracy = (predictions == y_test.to(device)).float().mean()
    print(f"Final Transformer Accuracy: {accuracy.item()*100:.2f}%")
    
# Sample prediction
idx = 10
test_post = X_raw[idx][:200]
true_cat = categories[y_raw[idx]]
pred_cat = categories[predictions[idx]]
print(f"\nPost Content: {test_post}...")
print(f"True: {true_cat} | Predicted: {pred_cat}")

Final Transformer Accuracy: 71.24%

Post Content: 
A question for you - can you give me the name of an organization or a
philosophy or a political movement, etc., which has never had anything
evil done in its name?  You're missing a central teaching ...
True: sci.med | Predicted: soc.religion.christian
